In [ ]:
!pip install xgboost

In [ ]:
!pip install --upgrade gspread

In [ ]:
!pip install gspread_formatting

In [ ]:
!pip install nltk

In [ ]:
from google.colab import auth
auth.authenticate_user()


import calendar
from datetime import datetime
import pandas as pd
import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)
# Libros
libro = gc.open_by_key("1r6mOjvi5ITuIgVJn9ec6hNph_ECWywA90uJxZnNYYCY") #Original

In [ ]:
# SolicitudDeInversi?nSocialFocal=libro.get_worksheet(1)
# TablaCausas=libro.get_worksheet(9)
# TablaEfectos=libro.get_worksheet(8)
# Actividades=libro.get_worksheet(5)
# CoinversoresTabla=libro.get_worksheet(11)
# AliadosDelProyecto=libro.get_worksheet(2)

In [ ]:
TablaFinImpacto=libro.get_worksheet(7)
TablaFinImpacto_d = TablaFinImpacto.get_all_values()
TablaFinImpacto_df=pd.DataFrame(TablaFinImpacto_d)
TablaFinImpacto_df=TablaFinImpacto_df.iloc[2:,[0,4]]
TablaFinImpacto_df.columns=["ID", "Respuesta"]
TablaFinImpacto_df

In [ ]:
C?lculo_Puntaje=gc.open_by_key("1mqxPzWLSjj13hhM4VVJRTWGMsNI1R4-tHgFAbO81MJg") #Original
Rubrica=C?lculo_Puntaje.get_worksheet(1)
Rubrica_d = Rubrica.get_all_values()
Rubrica_df=pd.DataFrame(Rubrica_d)
Rubrica_df=Rubrica_df.iloc[1:,[0,1]]
Rubrica_df.columns=["Mision", "Clasificaci?n"]
Rubrica_df["Mision"]=Rubrica_df["Mision"].astype(str)
Rubrica_df['Clasificaci?n'] = range(0, len(Rubrica_df))
Rubrica_df

In [ ]:
Puntajes=gc.open_by_key("1cXhgAQDmUhOlgHqjkLghYbPtqY7MqE66H2g5VUt_Mhs")
Puntaje=Puntajes.get_worksheet(0)
Puntaje_d = Puntaje.get_all_values()
Puntaje_df=pd.DataFrame(Puntaje_d)
Puntaje_df=Puntaje_df.iloc[2:,[0,1]]
Puntaje_df.columns=["ID","Mision"]
Puntaje_df

In [ ]:
acentos = {'?': 'a', '?': 'e', '?': 'i', '?': 'o', '?': 'u',
           '?': 'A', '?': 'E', '?': 'I', '?': 'O', '?': 'U', "?":"N", "?":"n"}

def quitar_acentos(texto):
    for acentuada, sin_acento in acentos.items():
        texto = texto.replace(acentuada, sin_acento)
    return texto
df_sfinal = pd.merge(Puntaje_df,Rubrica_df, on='Mision', how='inner')
df_final= pd.merge(df_sfinal,TablaFinImpacto_df, on="ID", how="inner")
df_final=df_final.iloc[:,[3,2]]
df_final['Respuesta'] = df_final['Respuesta'].apply(quitar_acentos)
df_final['Respuesta'] = df_final['Respuesta'].astype(str)
df_final['Clasificaci?n'] = df_final['Clasificaci?n'].apply(lambda x: x - 2)
df_final

In [ ]:
import nltk

nltk.download('all')
text = list(df_final['Respuesta'])
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
lemmatizer = WordNetLemmatizer()
corpus = []
for i in range(len(text)):

    r = re.sub('[^a-zA-Z]', ' ', text[i])

    r = r.lower()

    r = r.split()

    r = [word for word in r if word not in stopwords.words('spanish')]

    r = [lemmatizer.lemmatize(word) for word in r]

    r = ' '.join(r)

    corpus.append(r)




#assign corpus to data['text']

df_final['Respuesta'] = corpus

# data.head()

In [ ]:
# Create Feature and Label sets

X = df_final['Respuesta']

y = df_final['Clasificaci?n']




# train test split (66% train - 33% test)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

#ANN

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=1000)  # 'num_words' define el n?mero m?ximo de palabras que queremos considerar
tokenizer.fit_on_texts(X_train)  # Ajustamos el tokenizer al conjunto de entrenamiento

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Rellenar las secuencias para que todas tengan la misma longitud
maxlen = 100  # Puedes ajustar este valor dependiendo de la longitud promedio de tus textos
X_train_pad = pad_sequences(X_train_seq, padding='post', maxlen=maxlen)
X_test_pad = pad_sequences(X_test_seq, padding='post', maxlen=maxlen)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense


model = Sequential()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.optimizers import Adam

# Crear el modelo LSTM
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=128, input_length=X_train_pad.shape[1]))

model.add(LSTM(128, return_sequences=True))  # 'return_sequences=True' para devolver la salida de cada celda LSTM
model.add(Dropout(0.5))  # Capa Dropout para prevenir overfitting

model.add(LSTM(128, return_sequences=True))  # Devolvemos secuencias para las siguientes capas LSTM
model.add(Dropout(0.5))

model.add(LSTM(64, return_sequences=False))  # ?ltima capa LSTM, no necesitamos devolver secuencias
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))  # Esta capa es opcional, pero puede mejorar el rendimiento


num_classes = 5  # Para clasificaci?n multiclase
model.add(Dense(num_classes, activation='softmax'))  # Usamos 'softmax' para clasificaci?n multiclase

model.compile(loss='sparse_categorical_crossentropy',  # Usa 'binary_crossentropy' si es clasificaci?n binaria
              optimizer=Adam(),
              metrics=['accuracy'])

# Ver el resumen del modelo
model.summary()


In [ ]:
history = model.fit(X_train_pad, y_train, epochs=100, batch_size=32, validation_data=(X_test_pad, y_test))

#Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),  # Convert text to numerical vectors using TF-IDF
    ('nb', MultinomialNB())  # Use KNN classifier to classify the data
])

parameters = {
    'tfidf__max_df': [0.5, 0.75, 1.0],
    'tfidf__ngram_range': [(1,1), (1,2)],
    'nb__alpha': [0.1, 0.5, 1, 2, 5]
}

grid_search = GridSearchCV(pipeline, parameters, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_
predictions_MNB = best_pipeline.predict(X_test)

# imprimimos el reporte de la clasificaci?n
# print(classification_report(y_test, predictions_MNB))

In [ ]:
# # Importaci?n de la librer?a necesaria
# from sklearn.naive_bayes import MultinomialNB

# # Crear el modelo Naive Bayes
# nb = MultinomialNB()

# # Entrenar el modelo utilizando los datos de entrenamiento
# nb.fit(X_train_cv, y_train)


# # Generar las predicciones
# predictions_MNB = nb.predict(X_test_cv)

# # Mostrar las predicciones
# predictions_MNB

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report, roc_curve, auc, log_loss, matthews_corrcoef

accuracy_MNB = accuracy_score(y_test, predictions_MNB)

conf_matrix_MNB = confusion_matrix(y_test, predictions_MNB)

precision_MNB = precision_score(y_test, predictions_MNB, average='weighted')  # 'weighted' es ?til en clasificaci?n multiclase

recall_MNB = recall_score(y_test, predictions_MNB, average='weighted')  # 'weighted' es ?til en clasificaci?n multiclase

f1_MNB = f1_score(y_test, predictions_MNB, average='weighted')  # 'weighted' es ?til en clasificaci?n multiclase

report = classification_report(y_test, predictions_MNB)

print(f'Accuracy: {accuracy_MNB:.4f}')
print('Confusion Matrix:')
print(conf_matrix_MNB)
print(f'Precision (weighted): {precision_MNB:.4f}')
print(f'Recall (weighted): {recall_MNB:.4f}')
print(f'F1-Score (weighted): {f1_MNB:.4f}')
print('Classification Report:')
print(report)


In [ ]:
import pickle
filename = "NBbueno.pkl"
outfile = open(filename, 'wb')

pickle.dump(best_pipeline, outfile, 2)
outfile.close()

## XGB boost

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)


X_train_cv.shape

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

model.fit(X_train_cv, y_train)

predictions_XGB = model.predict(X_test_cv)

In [ ]:
accuracy_XGB = accuracy_score(y_test, predictions_XGB)

conf_matrix_XGB = confusion_matrix(y_test, predictions_XGB)

precision_XGB = precision_score(y_test, predictions_XGB, average='weighted')  # 'weighted' es ?til en clasificaci?n multiclase

recall_XGB = recall_score(y_test, predictions_XGB, average='weighted')  # 'weighted' es ?til en clasificaci?n multiclase

f1_XGB = f1_score(y_test, predictions_XGB, average='weighted')  # 'weighted' es ?til en clasificaci?n multiclase

report_XGB = classification_report(y_test, predictions_XGB)

print(f'Accuracy: {accuracy_XGB:.4f}')
print('Confusion Matrix:')
print(conf_matrix_XGB)
print(f'Precision (weighted): {precision_XGB:.4f}')
print(f'Recall (weighted): {recall_XGB:.4f}')
print(f'F1-Score (weighted): {f1_XGB:.4f}')
print('Classification Report:')
print(report_XGB)
